# Hands-On Lab: K-Means, K-Medoids (PAM) & Silhouette Analysis
Accompanies **Chapter 20: K-Means and Partitioning Around Medoids (PAM)**.

In this lab, we:
1. Implement the **Elbow Method** (WCSS Inertia curve) to locate diminishing returns.
2. Calculate and plot **Silhouette Coefficients** to determine the optimal cluster count $K$.
3. Benchmark **K-Means vs. K-Medoids (PAM)** on outlier-contaminated data.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

data_path = os.path.abspath('../../data/bd_train.csv')
df = pd.read_csv(data_path)

features = ['age', 'balance', 'duration', 'campaign']
X = df[features].dropna().sample(n=2000, random_state=42)
X_scaled = StandardScaler().fit_transform(X)

k_range = range(2, 8)
inertias, sil_scores = [], []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42).fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(k_range, inertias, marker='o', color='royalblue', linewidth=2)
ax1.set_title('The Elbow Method (Inertia)', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(k_range, sil_scores, marker='s', color='forestgreen', linewidth=2)
ax2.set_title('Silhouette Score Sweep', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Outlier Demonstration
np.random.seed(42)
clean_pts = np.random.normal(loc=0.0, scale=1.0, size=(80, 2))
contaminated_pts = np.vstack([clean_pts, [50.0, 50.0]])

km_center = KMeans(n_clusters=1, n_init=10, random_state=42).fit(contaminated_pts).cluster_centers_[0]
diff = contaminated_pts[:, np.newaxis, :] - contaminated_pts[np.newaxis, :, :]
D = np.linalg.norm(diff, axis=-1)
medoid_center = contaminated_pts[np.argmin(np.sum(D, axis=1))]

print(f"K-Means Centroid: [{km_center[0]:.2f}, {km_center[1]:.2f}]  (Pulled by outlier!)")
print(f"K-Medoid Medoid:  [{medoid_center[0]:.2f}, {medoid_center[1]:.2f}]  (Anchored to cluster!)")